# Bangladesh Passport Virtual Consular Officer (CrewAI)

This notebook implements a multi-agent passport readiness assistant with:
- `Context7`/`Apify` planning checkpoints
- scrape-then-fallback data pipeline
- policy, fee, and document agents
- English markdown table + Bangla summary output
- scenario tests and report exports

## 1) Workspace & Virtual Environment Verification (VS Code + current `venv`)

In [2]:
from pathlib import Path
import os
import sys
import json
from datetime import datetime

PROJECT_ROOT = Path.cwd()
print('Project root:', PROJECT_ROOT)
print('Python executable:', sys.executable)
print('Virtual env:', os.environ.get('VIRTUAL_ENV', '(not set in kernel env)'))
print('Using project venv kernel:', 'venv' in sys.executable.lower())

Project root: d:\Study\Ostad\18. Multi-Agent Systems and Workflow Automation\BD_Passport_AI_Agent
Python executable: d:\Study\Ostad\18. Multi-Agent Systems and Workflow Automation\BD_Passport_AI_Agent\venv\Scripts\python.exe
Virtual env: D:\Study\Ostad\18. Multi-Agent Systems and Workflow Automation\BD_Passport_AI_Agent\venv
Using project venv kernel: True


## 2) Update `requirements.txt` for MCP + scraping support (latest versions only)

In [3]:
import importlib.util

requirements_path = PROJECT_ROOT / 'requirements.txt'
required_packages = [
    'crewai',
    'crewai-tools',
    'litellm',
    'python-dotenv',
    'ipykernel',
    'requests',
    'beautifulsoup4',
]

if requirements_path.exists():
    current = [line.strip() for line in requirements_path.read_text(encoding='utf-8').splitlines() if line.strip()]
else:
    current = []

missing_lines = [pkg for pkg in required_packages if pkg not in current]
print('requirements.txt path:', requirements_path)
print('Current lines:', current)
print('Suggested append lines (latest/no pin):', missing_lines)

requirements.txt path: d:\Study\Ostad\18. Multi-Agent Systems and Workflow Automation\BD_Passport_AI_Agent\requirements.txt
Current lines: ['crewai', 'crewai-tools', 'litellm', 'python-dotenv', 'ipykernel', 'requests', 'beautifulsoup4']
Suggested append lines (latest/no pin): []


## 3) Install/Sync Dependencies in Active `venv`

In [4]:
import subprocess

print('Running pip sync commands in active kernel interpreter...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=False)

for mod_name in ['crewai', 'crewai_tools', 'litellm', 'dotenv', 'requests', 'bs4']:
    spec = importlib.util.find_spec(mod_name)
    print(f'{mod_name}:', 'OK' if spec else 'MISSING')

Running pip sync commands in active kernel interpreter...
crewai: OK
crewai_tools: OK
litellm: OK
dotenv: OK
requests: OK
bs4: OK


## 4) Environment Configuration with `.env` (provider switching: Ollama/Gemini)

In [5]:
from dotenv import load_dotenv
from crewai import LLM

load_dotenv(PROJECT_ROOT / '.env')

def mask_secret(value: str | None) -> str:
    if not value:
        return '(missing)'
    if len(value) <= 8:
        return '*' * len(value)
    return value[:4] + '...' + value[-4:]

LLM_PROVIDER = os.getenv('LLM_PROVIDER', 'ollama').strip().lower()
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GEMINI_MODEL = os.getenv('GEMINI_MODEL', 'gemini/gemini-2.5-flash')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'ollama/qwen2.5:7b')

print('LLM_PROVIDER:', LLM_PROVIDER)
print('GEMINI_MODEL:', GEMINI_MODEL)
print('GEMINI_API_KEY:', mask_secret(GEMINI_API_KEY))
print('OLLAMA_BASE_URL:', OLLAMA_BASE_URL)
print('OLLAMA_MODEL:', OLLAMA_MODEL)

def build_llm() -> LLM:
    if LLM_PROVIDER == 'gemini':
        if not GEMINI_API_KEY:
            raise ValueError('GEMINI_API_KEY is missing while LLM_PROVIDER=gemini')
        return LLM(model=GEMINI_MODEL, api_key=GEMINI_API_KEY, temperature=0.2)
    if LLM_PROVIDER == 'ollama':
        return LLM(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.2)
    raise ValueError("LLM_PROVIDER must be 'gemini' or 'ollama'")

selected_llm = build_llm()
print('LLM ready:', selected_llm.model)

LLM_PROVIDER: ollama
GEMINI_MODEL: gemini/gemini-2.5-flash
GEMINI_API_KEY: AIza...lu7w
OLLAMA_BASE_URL: http://localhost:11434
OLLAMA_MODEL: ollama/qwen2.5:7b
LLM ready: ollama/qwen2.5:7b


## 5) MCP Workflow Plan: Context7 + Apify (step-by-step orchestration)

In [6]:
mcp_plan = {
    'step_1': 'Use Context7 to verify latest CrewAI patterns for Agent/Task/Crew/context usage.',
    'step_2': 'Use Apify + HTTP fetch to collect passport policy/fee evidence from official portal.',
    'step_3': 'Implement scrape-then-fallback functions with explicit source metadata.',
    'step_4': 'Build CrewAI agents and delegate tasks Policy -> Fee -> Documents -> Final Report.',
    'step_5': 'Run scenario tests, assert inconsistency flags, and export markdown artifacts.'
}
print(json.dumps(mcp_plan, indent=2, ensure_ascii=False))

{
  "step_1": "Use Context7 to verify latest CrewAI patterns for Agent/Task/Crew/context usage.",
  "step_2": "Use Apify + HTTP fetch to collect passport policy/fee evidence from official portal.",
  "step_3": "Implement scrape-then-fallback functions with explicit source metadata.",
  "step_4": "Build CrewAI agents and delegate tasks Policy -> Fee -> Documents -> Final Report.",
  "step_5": "Run scenario tests, assert inconsistency flags, and export markdown artifacts."
}


## 6) Local Fallback Database (`passport_rules.json`) for fees and documents

In [7]:
passport_rules_path = PROJECT_ROOT / 'passport_rules.json'

default_rules = {
    'fees_2026': {
        '48_pages': {
            '5_years': {'regular': 4025, 'express': 6325, 'super_express': 8625},
            '10_years': {'regular': 5750, 'express': 8050, 'super_express': 10350}
        },
        '64_pages': {
            '5_years': {'regular': 6325, 'express': 8625, 'super_express': 12075},
            '10_years': {'regular': 8050, 'express': 10350, 'super_express': 13800}
        }
    },
    'required_docs': {
        'adult': ['NID Card', 'Application Summary', 'Payment Slip'],
        'minor_under_18': ['Birth Registration (English)', 'Parents NID', '3R Photo'],
        'government_staff': ['NOC (No Objection Certificate)', 'NID'],
        'name_change': ['Marriage Certificate / Nikahnama / Divorce Deed (if applicable)'],
        'profession_proof': ['Employment Certificate / Trade License / Student ID (as applicable)']
    }
}

if not passport_rules_path.exists():
    passport_rules_path.write_text(json.dumps(default_rules, ensure_ascii=False, indent=2), encoding='utf-8')

local_rules = json.loads(passport_rules_path.read_text(encoding='utf-8'))

if 'fees_2026' not in local_rules:
    local_rules['fees_2026'] = default_rules['fees_2026']
if 'required_docs' not in local_rules:
    local_rules['required_docs'] = default_rules['required_docs']

print('Local DB loaded from:', passport_rules_path)
print('Top-level keys:', list(local_rules.keys()))

Local DB loaded from: d:\Study\Ostad\18. Multi-Agent Systems and Workflow Automation\BD_Passport_AI_Agent\passport_rules.json
Top-level keys: ['fees_2026', 'required_docs']


## 7) User Profile Schema & Input Normalization

In [8]:
from dataclasses import dataclass, asdict

DELIVERY_MAP = {
    'regular': 'regular',
    'normal': 'regular',
    'express': 'express',
    'urgent': 'express',
    'super express': 'super_express',
    'super_express': 'super_express',
    'super-urgent': 'super_express'
}

@dataclass
class UserProfile:
    age: int
    profession: str
    urgency: str
    page_count: int
    requested_validity_years: int | None = None
    has_nid: bool = False
    location: str = 'Bangladesh'
    needs_name_change: bool = False


def normalize_profile(raw: dict) -> UserProfile:
    age = int(raw['age'])
    profession = str(raw.get('profession', 'unknown')).strip().lower()
    urgency_raw = str(raw.get('urgency', 'regular')).strip().lower()
    urgency = DELIVERY_MAP.get(urgency_raw, 'regular')
    page_count = int(raw.get('page_count', 48))
    if page_count not in (48, 64):
        page_count = 48
    requested_validity_years = raw.get('requested_validity_years')
    if requested_validity_years is not None:
        requested_validity_years = int(requested_validity_years)
    has_nid = bool(raw.get('has_nid', False))
    location = str(raw.get('location', 'Bangladesh')).strip()
    needs_name_change = bool(raw.get('needs_name_change', False))
    return UserProfile(
        age=age,
        profession=profession,
        urgency=urgency,
        page_count=page_count,
        requested_validity_years=requested_validity_years,
        has_nid=has_nid,
        location=location,
        needs_name_change=needs_name_change,
    )

## 8) Policy Guardian Agent (eligibility, age constraints, ID rules)

In [9]:
from crewai import Agent, Task, Crew, Process


def policy_rules(profile: UserProfile) -> dict:
    flags = []
    if profile.age < 18:
        allowed_validity_years = [5]
        required_id = 'Birth Registration (English)'
        page_constraint = 48
    elif profile.age > 65:
        allowed_validity_years = [5, 10]
        required_id = 'NID (preferred) or Birth Registration as per policy context'
        page_constraint = None
    elif profile.age < 20:
        allowed_validity_years = [5, 10]
        required_id = 'NID or Birth Registration (English)'
        page_constraint = None
    else:
        allowed_validity_years = [5, 10]
        required_id = 'NID (mandatory)'
        page_constraint = None

    recommended_validity = 10 if 10 in allowed_validity_years else 5

    if profile.requested_validity_years and profile.requested_validity_years not in allowed_validity_years:
        flags.append(
            f"Inconsistency: age {profile.age} cannot request {profile.requested_validity_years}-year validity."
        )

    if page_constraint and profile.page_count != page_constraint:
        flags.append(f'Under-18 applicants are restricted to {page_constraint} pages; requested {profile.page_count}.')

    if profile.age >= 20 and not profile.has_nid:
        flags.append('NID is required for applicants above 20 for local applications.')

    return {
        'allowed_validity_years': allowed_validity_years,
        'required_id': required_id,
        'recommended_validity': recommended_validity,
        'flags': flags,
    }

policy_guardian = Agent(
    role='Policy Guardian (Bangladesh Passport Policy Expert)',
    goal='Determine applicant eligibility, allowed passport validity, and correct identity document requirements from age and policy.',
    backstory='A senior consular policy analyst who reviews Bangladesh e-passport rules for age brackets and legal ID requirements.',
    llm=selected_llm,
    verbose=True,
    allow_delegation=False,
)

## 9) Fee Calculator Agent (2026 fee lookup + VAT validation + inconsistency flagging)

In [10]:
def compute_fee(profile: UserProfile, policy_output: dict, fee_source: dict) -> dict:
    flags = list(policy_output.get('flags', []))
    fees = fee_source.get('fees_2026', {})

    validity = profile.requested_validity_years or policy_output['recommended_validity']
    if validity not in policy_output['allowed_validity_years']:
        flags.append(f'Fee blocked: requested validity {validity} is not permitted for this applicant.')
        validity = policy_output['recommended_validity']

    page_key = f'{profile.page_count}_pages'
    validity_key = f'{validity}_years'
    delivery_key = profile.urgency

    base_fee = fees.get(page_key, {}).get(validity_key, {}).get(delivery_key)
    if base_fee is None:
        flags.append('Fee lookup failed in selected source; fallback to default local DB structure.')
        base_fee = default_rules['fees_2026'][page_key][validity_key][delivery_key]

    vat_rate = 0.15
    vat_component = round(base_fee * vat_rate)
    total_fee = int(base_fee)

    if abs(total_fee - round(base_fee)) != 0:
        flags.append('Fee total format mismatch detected.')

    return {
        'validity_years': validity,
        'delivery_type': delivery_key,
        'base_without_vat': round(base_fee / (1 + vat_rate), 2),
        'vat_rate': vat_rate,
        'vat_component_estimate': vat_component,
        'total_fee_bdt': total_fee,
        'flags': flags,
    }

chancellor = Agent(
    role='Chancellor of the Exchequer (Financial Auditor)',
    goal='Calculate exact e-passport fee from official fee structure and validate VAT-consistent totals.',
    backstory='A meticulous government fee auditor who verifies passport charges, delivery surcharges, and consistency flags.',
    llm=selected_llm,
    verbose=True,
    allow_delegation=False,
)

## 10) Document Architect Agent (dynamic checklist by profession/status)

In [11]:
def build_checklist(profile: UserProfile, docs_source: dict) -> list[str]:
    docs_cfg = docs_source.get('required_docs', {})
    checklist = []

    if profile.age < 18:
        checklist.extend(docs_cfg.get('minor_under_18', []))
    else:
        checklist.extend(docs_cfg.get('adult', []))

    if 'gov' in profile.profession or 'government' in profile.profession:
        checklist.extend(docs_cfg.get('government_staff', []))

    if profile.needs_name_change:
        checklist.extend(docs_cfg.get('name_change', []))

    checklist.extend(docs_cfg.get('profession_proof', []))

    deduped = []
    seen = set()
    for item in checklist:
        key = item.strip().lower()
        if key and key not in seen:
            seen.add(key)
            deduped.append(item)
    return deduped

document_architect = Agent(
    role='Document Architect (Documentation Officer)',
    goal='Generate a customized, applicant-specific passport document checklist.',
    backstory='A documentation specialist in a consular office known for preparing complete and error-free application bundles.',
    llm=selected_llm,
    verbose=True,
    allow_delegation=False,
)

## 11) Crew Task Graph & Delegation (Policy output as Fee input context)

In [12]:
def build_crew(profile: UserProfile, source_metadata: dict) -> tuple[Crew, dict]:
    policy_expected = (
        'JSON with allowed_validity_years, required_id, recommended_validity, flags. '
        'Flag impossible age-validity requests.'
    )
    fee_expected = (
        'JSON with validity_years, delivery_type, total_fee_bdt, VAT explanation, flags. '
        'Use Policy Guardian output as strict context.'
    )
    docs_expected = 'JSON with checklist array and notes for profession or special status.'
    final_expected = 'English markdown table + Bangla summary + data source (scrape/fallback).'

    policy_task = Task(
        description=(
            f'Applicant profile: {asdict(profile)}. '
            'Determine validity eligibility and identification requirements by age policy. '
            'If under 18, enforce 5-year and 48-page constraints.'
        ),
        expected_output=policy_expected,
        agent=policy_guardian,
    )

    fee_task = Task(
        description=(
            'Using Policy Guardian output as context, compute inside-Bangladesh fee for 2026 structure '
            f'for {profile.page_count} pages and delivery={profile.urgency}. Validate consistency and flag conflicts.'
        ),
        expected_output=fee_expected,
        agent=chancellor,
        context=[policy_task],
    )

    doc_task = Task(
        description=(
            'Build checklist for this applicant including age, profession and optional name-change evidence. '
            'Deduplicate and keep concise.'
        ),
        expected_output=docs_expected,
        agent=document_architect,
        context=[policy_task],
    )

    final_task = Task(
        description=(
            'Produce final readiness report as an English markdown table plus Bangla summary. '
            f'Include source metadata: {source_metadata} and all inconsistency flags.'
        ),
        expected_output=final_expected,
        agent=document_architect,
        context=[policy_task, fee_task, doc_task],
    )

    crew = Crew(
        agents=[policy_guardian, chancellor, document_architect],
        tasks=[policy_task, fee_task, doc_task, final_task],
        process=Process.sequential,
        verbose=True,
    )

    return crew, {
        'policy_task': policy_task,
        'fee_task': fee_task,
        'doc_task': doc_task,
        'final_task': final_task,
    }

## 12) Scrape-then-Fallback Pipeline (Apify/portal fetch with resilient error handling)

In [13]:
import re
import requests
from bs4 import BeautifulSoup


def scrape_fees_from_portal(timeout: int = 20) -> tuple[dict, dict]:
    url = 'https://www.epassport.gov.bd/instructions/passport-fees'
    metadata = {'source': 'scrape', 'url': url, 'error': None}
    try:
        resp = requests.get(url, timeout=timeout)
        resp.raise_for_status()
        text = BeautifulSoup(resp.text, 'html.parser').get_text(' ', strip=True)

        patterns = {
            ('48_pages', '5_years', 'regular'): r'48 pages and 5 years validity.*?Regular delivery:\s*TK\s*([\d,]+)',
            ('48_pages', '5_years', 'express'): r'48 pages and 5 years validity.*?Express delivery:\s*TK\s*([\d,]+)',
            ('48_pages', '5_years', 'super_express'): r'48 pages and 5 years validity.*?Super Express delivery:\s*TK\s*([\d,]+)',
            ('48_pages', '10_years', 'regular'): r'48 pages and 10 years validity.*?Regular delivery:\s*TK\s*([\d,]+)',
            ('48_pages', '10_years', 'express'): r'48 pages and 10 years validity.*?Express delivery:\s*TK\s*([\d,]+)',
            ('48_pages', '10_years', 'super_express'): r'48 pages and 10 years validity.*?Super Express delivery:\s*TK\s*([\d,]+)',
            ('64_pages', '5_years', 'regular'): r'64 pages and 5 years validity.*?Regular delivery:\s*TK\s*([\d,]+)',
            ('64_pages', '5_years', 'express'): r'64 pages and 5 years validity.*?Express delivery:\s*TK\s*([\d,]+)',
            ('64_pages', '5_years', 'super_express'): r'64 pages and 5 years validity.*?Super Express delivery:\s*TK\s*([\d,]+)',
            ('64_pages', '10_years', 'regular'): r'64 pages and 10 years validity.*?Regular delivery:\s*TK\s*([\d,]+)',
            ('64_pages', '10_years', 'express'): r'64 pages and 10 years validity.*?Express delivery:\s*TK\s*([\d,]+)',
            ('64_pages', '10_years', 'super_express'): r'64 pages and 10 years validity.*?Super Express delivery:\s*TK\s*([\d,]+)',
        }

        parsed = {'fees_2026': {'48_pages': {'5_years': {}, '10_years': {}}, '64_pages': {'5_years': {}, '10_years': {}}}}

        for (p, v, d), pattern in patterns.items():
            match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
            if match:
                parsed['fees_2026'][p][v][d] = int(match.group(1).replace(',', ''))

        enough_data = all(
            key in parsed['fees_2026']['48_pages']['5_years']
            for key in ['regular', 'express', 'super_express']
        )
        if not enough_data:
            raise ValueError('Could not parse complete fee table from portal text.')

        parsed['required_docs'] = local_rules.get('required_docs', default_rules['required_docs'])
        return parsed, metadata
    except Exception as exc:
        metadata['source'] = 'fallback_local_db'
        metadata['error'] = str(exc)
        return local_rules, metadata

knowledge_base, source_metadata = scrape_fees_from_portal()
print('Data source:', source_metadata)

Data source: {'source': 'fallback_local_db', 'url': 'https://www.epassport.gov.bd/instructions/passport-fees', 'error': '403 Client Error: Forbidden for url: https://www.epassport.gov.bd/instructions/passport-fees'}


## 13) Report Renderer (English Markdown Table + Bangla companion summary)

In [14]:
from IPython.display import Markdown, display


def render_report(profile: UserProfile, policy_output: dict, fee_output: dict, checklist: list[str], source_meta: dict) -> dict:
    flags = sorted(set(policy_output.get('flags', []) + fee_output.get('flags', [])))
    english_table = f"""
| Field | Value |
|---|---|
| Applicant Age | {profile.age} |
| Profession | {profile.profession} |
| Validity | {fee_output['validity_years']} Years |
| Delivery Type | {fee_output['delivery_type'].replace('_', ' ').title()} |
| Passport Pages | {profile.page_count} |
| Total Fee | {fee_output['total_fee_bdt']} BDT |
| Required Identification | {policy_output['required_id']} |
| Documents Needed | {', '.join(checklist)} |
| Flags | {('; '.join(flags) if flags else 'None')} |
| Data Source | {source_meta['source']} |
""".strip()

    bangla_summary = (
        '### বাংলা সারাংশ\n'
        f"- আবেদনকারীর বয়স: {profile.age}\n"
        f"- পাসপোর্টের মেয়াদ: {fee_output['validity_years']} বছর\n"
        f"- ডেলিভারি ধরন: {fee_output['delivery_type']}\n"
        f"- মোট ফি: {fee_output['total_fee_bdt']} টাকা\n"
        f"- প্রয়োজনীয় ডকুমেন্ট: {', '.join(checklist)}\n"
        f"- সতর্কতা: {('; '.join(flags) if flags else 'কোনো অসঙ্গতি নেই')}\n"
    )

    report = {
        'english_markdown_table': english_table,
        'bangla_summary': bangla_summary,
        'flags': flags,
        'source_metadata': source_meta,
    }
    display(Markdown(english_table))
    display(Markdown(bangla_summary))
    return report

## 14) Scenario Tests in Notebook (including 15-year-old requesting 10-year validity)

In [14]:
def run_case(raw_profile: dict, run_crew: bool = True) -> dict:
    profile = normalize_profile(raw_profile)

    crew_debug = {'kickoff_result': None, 'kickoff_error': None}
    if run_crew:
        try:
            crew, _ = build_crew(profile, source_metadata)
            kickoff_result = crew.kickoff()
            crew_debug['kickoff_result'] = str(kickoff_result)
        except Exception as exc:
            crew_debug['kickoff_error'] = str(exc)

    policy_output = policy_rules(profile)
    fee_output = compute_fee(profile, policy_output, knowledge_base)
    checklist = build_checklist(profile, knowledge_base)
    report = render_report(profile, policy_output, fee_output, checklist, source_metadata)
    report['profile'] = asdict(profile)
    report['crew_debug'] = crew_debug
    return report

scenarios = {
    'adult_private_urgent': {
        'age': 24,
        'profession': 'private sector employee',
        'urgency': 'urgent',
        'page_count': 64,
        'requested_validity_years': 10,
        'has_nid': True,
        'location': 'Dhaka',
    },
    'minor_invalid_10_year': {
        'age': 15,
        'profession': 'student',
        'urgency': 'express',
        'page_count': 48,
        'requested_validity_years': 10,
        'has_nid': False,
        'location': 'Dhaka',
    },
    'government_staff': {
        'age': 40,
        'profession': 'government staff',
        'urgency': 'regular',
        'page_count': 48,
        'requested_validity_years': 10,
        'has_nid': True,
        'location': 'Chattogram',
    },
    'missing_nid_adult': {
        'age': 28,
        'profession': 'private sector employee',
        'urgency': 'express',
        'page_count': 48,
        'requested_validity_years': 10,
        'has_nid': False,
        'location': 'Dhaka',
    }
}

reports = {}
for name, case in scenarios.items():
    print('\n' + '='*80)
    print('Scenario:', name)
    reports[name] = run_case(case, run_crew=True)

assert any('cannot request 10-year' in f.lower() for f in reports['minor_invalid_10_year']['flags'])
if source_metadata['source'] != 'scrape':
    assert source_metadata['source'] == 'fallback_local_db'
print('Scenario assertions passed.')


Scenario: adult_private_urgent


╭──────────────────────────────────────────────── Yanked Version ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Version 1.10.0 has been yanked from PyPI.                                                                      │
│  Reason: miss behaving when running on crewai AMP                                                               │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f5564122-d756-4645-84be-263a0dc9e23d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: {'age': 24, 'profession': 'private sector employee', 'urgency': 'express',            │
│  'page_count': 64, 'requested_validity_years': 10, 'has_nid': True, 'location': 'Dhaka', 'needs_name_change':   │
│  False}. Determine validity eligibility and identification requirements by age policy. If under 18, enforce     │
│  5-year and 48-page constraints.                                                                                │
│  ID: 11987315-ee50-4ac6-812f-dfddc3ea0998                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Policy Guardian (Bangladesh Passport Policy Expert)                                                     │
│                                                                                                                 │
│  Task: Applicant profile: {'age': 24, 'profession': 'private sector employee', 'urgency': 'express',            │
│  'page_count': 64, 'requested_validity_years': 10, 'has_nid': True, 'location': 'Dhaka', 'needs_name_change':   │
│  False}. Determine validity eligibility and identification requirements by age policy. If under 18, enforce     │
│  5-year and 48-page constraints.                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f2dfba59-d037-4d72-ac06-47eaf16b598a                                                                       │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "checklist": [                                                                                               │
│      {                                                                                                          │
│        "item": "Birth Certificate",                                                                             │
│        "notes": "Required identification document for applicants over 18 without a National Identity Card       │
│  (NID)."                                                                                                        │
│      },                                                                                                         │
│      {                                                                                                          │
│        "item": "Passport Application Form",                                                                     │
│        "notes": "Complete and sign the form."                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "item": "Two Recent Passport Photos",                                                                    │
│        "notes": "Ensure photos meet passport photo requirements."                                               │
│      },                                                                                                         │
│      {                                                                                                          │
│        "item": "Employment Verification",                                                                       │
│        "notes": "Optional, depending on profession. Not applicable if self-employed or retired."                │
│      },                                                                                                         │
│      {                                                                                                          │
│        "item": "Name Change Evidence (if applicable)",                                                          │
│        "notes": "Provide documentation of name change if the applicant has changed their name."                 │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "profession_notes": [                                                                                        │
│      "The applicant's profession is not specified, so no specific requirements apply. However, if               │
│  self-employed or in a special status, additional verification may be required."                                │
│    ]                                                                                                            │
│  }                                                    

| Field | Value |
|---|---|
| Applicant Age | 28 |
| Profession | private sector employee |
| Validity | 10 Years |
| Delivery Type | Express |
| Passport Pages | 48 |
| Total Fee | 8050 BDT |
| Required Identification | NID (mandatory) |
| Documents Needed | NID Card, Application Summary, Payment Slip, Employment Certificate / Trade License / Student ID (as applicable) |
| Flags | NID is required for applicants above 20 for local applications. |
| Data Source | fallback_local_db |

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### বাংলা সারাংশ
- আবেদনকারীর বয়স: 28
- পাসপোর্টের মেয়াদ: 10 বছর
- ডেলিভারি ধরন: express
- মোট ফি: 8050 টাকা
- প্রয়োজনীয় ডকুমেন্ট: NID Card, Application Summary, Payment Slip, Employment Certificate / Trade License / Student ID (as applicable)
- সতর্কতা: NID is required for applicants above 20 for local applications.


Scenario assertions passed.


## 14.1) Natural Language Input Showcase (LLM Agent → normalized JSON)

This demo cell accepts free-form user text, uses a dedicated CrewAI agent to extract structured profile JSON, and then sends that JSON to the same existing pipeline (`run_case`).

No hardcoded phrase matching is used for parsing in this showcase.

In [17]:
import json
import re


def extract_first_json_object(text: str) -> dict:
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        return json.loads(fenced.group(1))

    bare = re.search(r"(\{.*\})", text, flags=re.DOTALL)
    if bare:
        return json.loads(bare.group(1))

    raise ValueError('Could not find JSON object in interpreter output.')


def to_bool(value, default=False):
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        v = value.strip().lower()
        if v in {'true', 'yes', 'y', '1'}:
            return True
        if v in {'false', 'no', 'n', '0'}:
            return False
    return default


def interpret_profile_with_agent(user_text: str) -> dict:
    interpreter_agent = Agent(
        role='Profile Interpreter Agent',
        goal='Convert free-form user text about passport needs into a strict JSON profile schema.',
        backstory='A senior intake officer who standardizes applicant narratives into machine-readable fields for consular processing.',
        llm=selected_llm,
        verbose=True,
        allow_delegation=False,
    )

    schema_hint = {
        'age': 'int',
        'profession': 'string',
        'urgency': "one of ['regular','express','super_express']",
        'page_count': '48 or 64',
        'requested_validity_years': '5 or 10 or null',
        'has_nid': 'bool',
        'location': 'string',
        'needs_name_change': 'bool (default false if not mentioned)'
    }

    extraction_task = Task(
        description=(
            'Read the applicant message and return ONLY a valid JSON object with the required keys. '
            'Do not add explanations. Use null when value is not specified. '
            f'Required schema: {json.dumps(schema_hint, ensure_ascii=False)}\n\n'
            f'Applicant text:\n{user_text}'
        ),
        expected_output='A single strict JSON object and nothing else.',
        agent=interpreter_agent,
    )

    extraction_crew = Crew(
        agents=[interpreter_agent],
        tasks=[extraction_task],
        process=Process.sequential,
        verbose=True,
    )

    raw = str(extraction_crew.kickoff())
    parsed = extract_first_json_object(raw)

    normalized = {
        'age': int(parsed.get('age', 24)),
        'profession': str(parsed.get('profession', 'unknown')),
        'urgency': str(parsed.get('urgency', 'regular')),
        'page_count': int(parsed.get('page_count', 48)),
        'requested_validity_years': parsed.get('requested_validity_years', None),
        'has_nid': to_bool(parsed.get('has_nid', False), default=False),
        'location': str(parsed.get('location', 'Bangladesh')),
        'needs_name_change': to_bool(parsed.get('needs_name_change', False), default=False),
    }

    if normalized['requested_validity_years'] is not None:
        try:
            normalized['requested_validity_years'] = int(normalized['requested_validity_years'])
        except Exception:
            normalized['requested_validity_years'] = None

    return normalized


assignment_input_text = (
    'I am a 24-year-old private sector employee. '
    'I need a 64-page passport urgently because I have a business trip in two weeks. '
    'I have an NID and I live in Dhaka.'
)

parsed_profile = interpret_profile_with_agent(assignment_input_text)
print('Natural-language input:')
print(assignment_input_text)
print('\nAgent-interpreted profile JSON:')
print(json.dumps(parsed_profile, indent=2, ensure_ascii=False))

nl_report = run_case(parsed_profile, run_crew=True)


def get_table_value(markdown_table: str, field_name: str) -> str:
    for line in markdown_table.splitlines():
        if line.strip().startswith(f'| {field_name} |'):
            parts = [p.strip() for p in line.split('|')]
            if len(parts) >= 3:
                return parts[2]
    return '(not found)'


print('\nExpected Output Format (assignment-style):')
print('Validity:', get_table_value(nl_report['english_markdown_table'], 'Validity'))
print('Delivery Type:', get_table_value(nl_report['english_markdown_table'], 'Delivery Type'))
print('Total Fee:', get_table_value(nl_report['english_markdown_table'], 'Total Fee'))
print('Documents Needed:', get_table_value(nl_report['english_markdown_table'], 'Documents Needed'))

Natural-language input:
I am a 24-year-old private sector employee. I need a 64-page passport urgently because I have a business trip in two weeks. I have an NID and I live in Dhaka.

Parsed profile (internal JSON):
{
  "age": 24,
  "profession": "private sector employee",
  "urgency": "express",
  "page_count": 64,
  "requested_validity_years": null,
  "has_nid": true,
  "location": "Dhaka"
}


╭──────────────────────────────────────────────── Yanked Version ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Version 1.10.0 has been yanked from PyPI.                                                                      │
│  Reason: miss behaving when running on crewai AMP                                                               │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 897f3b2f-9494-47b4-925b-c87365d9c449                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: {'age': 24, 'profession': 'private sector employee', 'urgency': 'express',            │
│  'page_count': 64, 'requested_validity_years': None, 'has_nid': True, 'location': 'Dhaka',                      │
│  'needs_name_change': False}. Determine validity eligibility and identification requirements by age policy. If  │
│  under 18, enforce 5-year and 48-page constraints.                                                              │
│  ID: bf8ef31a-c9cb-4ac5-b896-7c0c88cb4ff1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Policy Guardian (Bangladesh Passport Policy Expert)                                                     │
│                                                                                                                 │
│  Task: Applicant profile: {'age': 24, 'profession': 'private sector employee', 'urgency': 'express',            │
│  'page_count': 64, 'requested_validity_years': None, 'has_nid': True, 'location': 'Dhaka',                      │
│  'needs_name_change': False}. Determine validity eligibility and identification requirements by age policy. If  │
│  under 18, enforce 5-year and 48-page constraints.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 897f3b2f-9494-47b4-925b-c87365d9c449                                                                       │
│  Final Output: ```markdown                                                                                      │
│  # Passport Application Checklist                                                                               │
│                                                                                                                 │
│  | Item | Notes |                                                                                               │
│  | --- | --- |                                                                                                  │
│  | National Identity Card (NID) | Required for identification. |                                                │
│  | Passport Application Form | Complete with accurate personal information. |                                   │
│  | Two Recent Passport Photos | Meet the size and quality requirements. |                                       │
│  | Birth Certificate | Required for verification of age and name. |                                             │
│  | Proof of Employment or Profession | Optional but recommended to include. |                                   │
│  | Name Change Evidence (if applicable) | Provide documents showing the change, such as court orders or         │
│  marriage certificates. |                                                                                       │
│                                                                                                                 │
│  ## Profession Notes                                                                                            │
│  For applicants in specific professions, additional documentation may be required. Please consult with the      │
│  consular office for detailed requirements.                                                                     │
│  ```                                                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # পাসপোর্ট আplিকেশন তালিকা                                                                                             │
│                                                                                                                 │
│  | বিষয় | নীতি |                                                                                                   │
│  | --- | --- |                                                                                                  │
│  | জাতীয় আইডেনটিফিকেশন কার্ড (NID) | পরিচয়পত্রের জন্য অবশ্যক। |                                                              │
│  | পাসপোর্ট আplিকেশন ফরম | একটি ব্যক্তিগত তথ্যসহ পূরণ করা উচিত। |                                                            │
│  | দুইটি নতুন পাসপোর্ট ফটো | আকার এবং গুণাবলীর অনুযায়ী পূরণ করা উচিত। |                                                           │
│  | জন্ম সনদ | বয়স ও নামের পরীক্ষার জন্য অবশ্যক। |                                                                       │
│  | শغل বা কর্ম প্রমাণ (অপশনাল) | সুপরিচিত হওয়া উচিত, যদি এটি অবশ্যক। |                               

| Field | Value |
|---|---|
| Applicant Age | 24 |
| Profession | private sector employee |
| Validity | 10 Years |
| Delivery Type | Express |
| Passport Pages | 64 |
| Total Fee | 10350 BDT |
| Required Identification | NID (mandatory) |
| Documents Needed | NID Card, Application Summary, Payment Slip, Employment Certificate / Trade License / Student ID (as applicable) |
| Flags | None |
| Data Source | fallback_local_db |

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### বাংলা সারাংশ
- আবেদনকারীর বয়স: 24
- পাসপোর্টের মেয়াদ: 10 বছর
- ডেলিভারি ধরন: express
- মোট ফি: 10350 টাকা
- প্রয়োজনীয় ডকুমেন্ট: NID Card, Application Summary, Payment Slip, Employment Certificate / Trade License / Student ID (as applicable)
- সতর্কতা: কোনো অসঙ্গতি নেই



Assignment-format summary:
Validity: 10 Years
Delivery Type: Express
Total Fee: 10350 BDT
Documents Needed: NID Card, Application Summary, Payment Slip, Employment Certificate / Trade License / Student ID (as applicable)


## 15) Export Artifacts (`passport_report.md`, `passport_report_error.md`)

In [15]:
ok_path = PROJECT_ROOT / 'passport_report.md'
err_path = PROJECT_ROOT / 'passport_report_error.md'

lines_ok = [f'# Passport Readiness Reports ({datetime.now().isoformat(timespec="seconds")})\n']
lines_err = [f'# Passport Flagged Reports ({datetime.now().isoformat(timespec="seconds")})\n']

for name, rep in reports.items():
    block = [
        f'## Scenario: {name}',
        rep['english_markdown_table'],
        rep['bangla_summary'],
        f"- Data source: {rep['source_metadata']}",
        '',
    ]
    has_flags = len(rep.get('flags', [])) > 0
    if has_flags:
        lines_err.extend(block)
    else:
        lines_ok.extend(block)

ok_path.write_text('\n'.join(lines_ok), encoding='utf-8')
err_path.write_text('\n'.join(lines_err), encoding='utf-8')

print('Wrote:', ok_path)
print('Wrote:', err_path)
print('Done.')

Wrote: d:\Study\Ostad\18. Multi-Agent Systems and Workflow Automation\BD_Passport_AI_Agent\passport_report.md
Wrote: d:\Study\Ostad\18. Multi-Agent Systems and Workflow Automation\BD_Passport_AI_Agent\passport_report_error.md
Done.
